### Netflix Movies and TV Shows
Listings of movies and tv shows on Netflix - Regularly Updated

In [6]:
import logging
import kagglehub  #pip install kagglehub
import shutil
from pathlib import Path
from sqlalchemy import create_engine
import pyodbc
import pandas as pd
from datetime import datetime

# Carpeta actual
base_path = Path.cwd()
print(base_path)

c:\Users\rfigu\Documents\portfoliosDev\NetflixMovies_TVShows


In [8]:
# CONFIGURACIÓN DEL LOGGER =========================
# Guardará una linea de log por cada pasa relevante del proceso ETL 
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.FileHandler(f"{base_path}/log/etl.log"),
        logging.StreamHandler()
    ]
)


logger = logging.getLogger("etl_kaggle_netflix")

In [3]:
# SETUP DE DIRECTORIOS
try:
    PROJECT_ROOT = Path.cwd()
    logger.info(f"Project root: {PROJECT_ROOT}")

    RAW_DIR = PROJECT_ROOT / "data" / "raw"
    STAGING_DIR = PROJECT_ROOT / "data" / "staging"
    PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

    for path in [RAW_DIR, STAGING_DIR, PROCESSED_DIR]:
        path.mkdir(parents=True, exist_ok=True)
        logger.info(f"Directorio verificado/creado: {path}")

except Exception as e:
    logger.exception("Error creando estructura de directorios")
    raise

# EXTRACT – KAGGLE
try:
    logger.info("Iniciando descarga del dataset desde Kaggle")

    download_path = Path(
        kagglehub.dataset_download("shivamb/netflix-shows")
    )

    logger.info(f"Dataset descargado en ruta temporal: {download_path}")

except Exception as e:
    logger.exception("Error durante la descarga del dataset")
    raise

# MOVE TO RAW Folder
try:
    logger.info("Moviendo archivos al directorio RAW")

    for file in download_path.iterdir():
        destination = RAW_DIR / file.name
        shutil.move(str(file), destination)
        logger.info(f"Archivo movido: {file.name} -> {destination}")

    logger.info(f"Proceso finalizado correctamente. RAW_DIR: {RAW_DIR}")

except Exception as e:
    logger.exception("Error moviendo archivos a RAW")
    raise


2026-01-11 17:42:11,084 | INFO | etl_kaggle_netflix | Project root: c:\Users\rfigu\Documents\portfoliosDev\NetflixMovies_TVShows
2026-01-11 17:42:11,090 | INFO | etl_kaggle_netflix | Directorio verificado/creado: c:\Users\rfigu\Documents\portfoliosDev\NetflixMovies_TVShows\data\raw
2026-01-11 17:42:11,092 | INFO | etl_kaggle_netflix | Directorio verificado/creado: c:\Users\rfigu\Documents\portfoliosDev\NetflixMovies_TVShows\data\staging
2026-01-11 17:42:11,092 | INFO | etl_kaggle_netflix | Directorio verificado/creado: c:\Users\rfigu\Documents\portfoliosDev\NetflixMovies_TVShows\data\processed
2026-01-11 17:42:11,092 | INFO | etl_kaggle_netflix | Iniciando descarga del dataset desde Kaggle


2026-01-11 17:42:12,574 | INFO | etl_kaggle_netflix | Dataset descargado en ruta temporal: C:\Users\rfigu\.cache\kagglehub\datasets\shivamb\netflix-shows\versions\5
2026-01-11 17:42:12,574 | INFO | etl_kaggle_netflix | Moviendo archivos al directorio RAW
2026-01-11 17:42:12,574 | INFO | etl_kaggle_netflix | Proceso finalizado correctamente. RAW_DIR: c:\Users\rfigu\Documents\portfoliosDev\NetflixMovies_TVShows\data\raw


In [9]:
# Parámetros de conexión
server = 'ASUSTUF\SQL22'  # Doble barra para escapar correctamente
database = 'NetflixTvSeries'

try:
    # logger.info(f"Intentando conectar a SQL Server: {server}, Base de datos: {database}")
    connP = pyodbc.connect(
        'DRIVER={ODBC Driver 17 for SQL Server};'
        f'SERVER={server};'
        f'DATABASE={database};'
        'Trusted_Connection=yes;'
    )
    
    logger.info("Conexión a con PyODDC establecida exitosamente.")
    print('Conexión exitosa')

except pyodbc.Error as e:
    logger.error(f"Error al conectar a la base de datos: {str(e)}")
    print(f'Error al conectar a la base de datos: {str(e)}')

2026-01-13 21:59:26,479 | INFO | etl_kaggle_netflix | Conexión a con PyODDC establecida exitosamente.


Conexión exitosa


In [5]:
print(RAW_DIR)

c:\Users\rfigu\Documents\portfoliosDev\NetflixMovies_TVShows\data\raw


In [6]:
try:
    logger.info("Lectura del archivo descargado")
    for file in RAW_DIR.iterdir():
        df = pd.read_csv(str(file))
    logger.info(f"Lectura correcta: {file}")
except Exception as e:
    logger.exception("Error al leer el archivo")
    raise
## se estadariza
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace(r"[^a-z0-9_]", "", regex=True)
)

df.head()


2026-01-11 17:42:12,792 | INFO | etl_kaggle_netflix | Lectura del archivo descargado
2026-01-11 17:42:12,894 | INFO | etl_kaggle_netflix | Lectura correcta: c:\Users\rfigu\Documents\portfoliosDev\NetflixMovies_TVShows\data\raw\netflix_titles.csv


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [7]:
len(df)

8807

In [8]:
# SQLAlchemy Parámetros de conexión
server = 'ASUSTUF\SQL22'
database = 'NetflixTvSeries'
driver = 'ODBC Driver 17 for SQL Server'

# Crear cadena de conexión para SQLAlchemy
# conexion_str = f"mssql+pyodbc://@{server}/{database}?driver={driver}&trusted_connection=yes"
conexion_str = f"mssql+pyodbc://@{server}/{database}?driver={driver}&trusted_connection=yes"

engine = create_engine(conexion_str)
conx=engine.connect

nombre_tabla = 'netflixTv_raw'

try:
    logger.info(f"Iniciando carga de hoja: {nombre_tabla}")
    # Eliminar filas completamente vacías
    df = df.dropna(how='all')
    # Agregar columna de fecha y hora de carga
    df['fecha_carga'] = datetime.now()
    # Insertar usando pandas y SQLAlchemy
    df.to_sql(name=nombre_tabla, con=engine, if_exists='replace', index=False)
              #,chunksize=500        muy importante)       
    ##engine.close()
    logger.info(f"Hoja '{nombre_tabla}' cargada exitosamente con {len(df)} filas.")
    print(f"✅ Hoja '{nombre_tabla}' cargada exitosamente con {len(df)} filas.")
    
except Exception as e:
    logger.error(f"Error al cargar hoja '{nombre_tabla}': {e}")
    print(f"❌ Error al cargar hoja '{nombre_tabla}': {e}")



2026-01-11 17:48:59,756 | INFO | etl_kaggle_netflix | Iniciando carga de hoja: netflixTv_raw
2026-01-11 17:49:07,627 | INFO | etl_kaggle_netflix | Hoja 'netflixTv_raw' cargada exitosamente con 8807 filas.


✅ Hoja 'netflixTv_raw' cargada exitosamente con 8807 filas.


In [9]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,fecha_carga
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2026-01-11 17:48:59.777688
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2026-01-11 17:48:59.777688
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2026-01-11 17:48:59.777688
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2026-01-11 17:48:59.777688
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2026-01-11 17:48:59.777688


In [ ]:
connP.close()
logger.info("Conexión a SQL cerrada")

In [10]:
# import pandas as pd

query = """
SELECT TOP 10 *
FROM dbo.netflixTv_raw
ORDER BY 1 DESC
"""
## utilizando la conexion de pyOdbc
df = pd.read_sql(query, connP)
df.head()

C:\Users\rfigu\AppData\Local\Temp\ipykernel_20940\1955445882.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connP)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,fecha_carga
0,s999,Movie,Searching For Sheela,None,Ma Anand Sheela,India,"April 22, 2021",2021,TV-14,58 min,"Documentaries, International Movies",Journalists and fans await Ma Anand Sheela as ...,2026-01-11 17:48:59.777
1,s998,TV Show,Life in Color with David Attenborough,None,David Attenborough,"Australia, United Kingdom","April 22, 2021",2021,TV-PG,1 Season,"British TV Shows, Docuseries, International TV...","Using innovative technology, this docuseries e...",2026-01-11 17:48:59.777
2,s997,Movie,HOMUNCULUS,Takashi Shimizu,"Go Ayano, Ryo Narita, Yukino Kishii, Anna Ishi...",Japan,"April 22, 2021",2021,TV-MA,116 min,"Horror Movies, International Movies, Thrillers",Truth and illusion blurs when a homeless amnes...,2026-01-11 17:48:59.777
3,s996,Movie,Vizontele,"Yilmaz Erdogan, Ömer Faruk Sorak","Yilmaz Erdogan, Demet Akbag, Altan Erkekli, Ce...",Turkey,"April 23, 2021",2001,TV-MA,106 min,"Comedies, Dramas, International Movies","In 1974, a rural town in Anatolia gets its fir...",2026-01-11 17:48:59.777
4,s995,Movie,This Lady Called Life,Kayode Kasum,"Bisola Aiyeola, Efa Iwara, Molawa Onajobi, Tin...",Nigeria,"April 23, 2021",2020,TV-14,120 min,"Dramas, International Movies, Romantic Movies","Abandoned by her family, young single mother A...",2026-01-11 17:48:59.777


### Exploracion de datos con errores (Junk)

In [14]:
df[df.show_id=='s995']

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,fecha_carga
4,s995,Movie,This Lady Called Life,Kayode Kasum,"Bisola Aiyeola, Efa Iwara, Molawa Onajobi, Tin...",Nigeria,"April 23, 2021",2020,TV-14,120 min,"Dramas, International Movies, Romantic Movies","Abandoned by her family, young single mother A...",2026-01-11 17:48:59.777
